In [ ]:
# 6.1 langgraph实现Agent基础操作     公里标

In [1]:
from typing import Literal #单位米
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END,StateGraph,MessagesState
from langgraph.prebuilt import ToolNode


#定义工具函数，用于Agent调用外部工具
@tool
def search(query:str):
    """模拟一个气象查询搜索工具"""
    if "北京" in query.lower() or "Beijing" in query.lower():
        return "阴天有雾，气温25度"
    return "天气晴朗温度较高39度"

#将工具函数存放在工具列表中
tools = [search]

C:\Users\wangz\miniconda3\envs\langchain\Lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [2]:
# 创建工具集节点：ToolNode是LangGraph中的一个预构建节点，用于封装一组工具函数。这些工具函数可以通过模型调用来执行特定的任务。

In [3]:
tool_node = ToolNode(tools)

In [6]:
#定义模型对象
API_KEY = open('./ken_files/deepseekAPI-Key.md',encoding='utf-8').read().strip()
model = ChatOpenAI(model_name="deepseek-chat",
                  api_key=API_KEY,base_url="https://api.deepseek.com")
#将工具列表绑定到模型对象上
model = model.bind_tools(tools)

In [5]:
def should_continue(state:MessagesState)->Literal["tools",END]:
    messages = state['messages']
    #获取用户提问消息
    last_message = messages[-1] 
    
    #如果llm调用工具，则转到tools节点
    if last_message.tool_calls:
        return "tools"
    return END

In [7]:
# 定义模型调用函数

In [8]:
def call_model(state:MessagesState):
    #获取消息列表
    messages = state['messages']
    #调用模型返回结果
    response = model.invoke(messages)
    return {"messages":[response]}

In [9]:
#定义一个新的状态图,使用MessagesState作为状态类型

In [10]:
workflow = StateGraph(MessagesState)

In [11]:
# 在状态图上添加节点

In [12]:
workflow.add_node("agent",call_model)
workflow.add_node("tools",tool_node)

In [13]:
# 设置入口节点为agent(入口节点指向agent节点)，这意味着agent是第一个被调用的节点

In [14]:
workflow.set_entry_point("agent")

In [15]:
# 添加条件边：agent节点根据should_continue进行边的连接（虚线边）

In [16]:
workflow.add_conditional_edges('agent',should_continue)

In [17]:
# 定义普通边:tools工具节点连接agent节点的边（实线边）

In [18]:
workflow.add_edge("tools","agent")

In [19]:
# 初始化内存以在图运行之间持久化状态：MemorySaver是LangGraph中的一个检查点保存器，用于在内存中保存状态图的中间状态。
# 这对于调试和监控非常有用，因为它允许你在运行时查看和恢复状态。

In [20]:
checkpointer = MemorySaver()

In [21]:
# 编译图：将其编译成一个langchain可运行的一个对象,在编译时传递内存

In [22]:
app = workflow.compile(checkpointer=checkpointer) 

In [23]:
# 执行图

In [24]:
final_state = app.invoke(
    {"messages":[HumanMessage(content="北京天气如何？")]},
    config={"configurable":{"thread_id":42}}
)
result = final_state['messages'][-1].content
result

'北京目前是阴天有雾，气温为25度。'

In [25]:
# 配置选项（config）实现上下文共享：如果两个任务在同一个线程上执行，它们可以共享同一个上下文（例如全局变量、线程本地存储等）。
# 这对于需要维护状态或会话信息的应用非常重要。

In [26]:
final_state = app.invoke(
    {"messages":[HumanMessage(content="我刚才问的是哪个城市？")]},
    config={"configurable":{"thread_id":42}}
)
result = final_state['messages'][-1].content
result

'你刚才询问的是**北京**的天气情况。'

In [27]:
# 保存图文件

In [28]:
graph_png = app.get_graph().draw_mermaid_png()
with open('graph.png','wb') as fp:
    fp.write(graph_png)

In [29]:
# 6.2 langgraph实现Multi-Agent Systems